In [1]:
!pip install groq python-dotenv

In [2]:
from dotenv import load_dotenv
import os
import getpass

# Load .env if present
load_dotenv()

# Prompt if API key not set
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


In [3]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [4]:
MODEL_NAME = "llama-3.3-70b-versatile"

MODEL_CONFIG = {
    "technical": {
        "system_prompt": """
You are a Technical Support Expert.
Be precise, code-focused, and analytical.
Provide step-by-step debugging guidance.
Include code snippets when helpful.
Avoid emotional language.
"""
    },

    "billing": {
        "system_prompt": """
You are a Billing Support Specialist.
Be empathetic and professional.
Focus on subscription policies, refunds, and transactions.
Explain next steps clearly.
"""
    },

    "general": {
        "system_prompt": """
You are a friendly general assistant.
Respond casually and helpfully.
Handle general conversation and non-technical questions.
"""
    }
}

In [5]:
def route_prompt(user_input):
    routing_prompt = f"""
Classify the following user query into one of these categories:
[technical, billing, general]

Return ONLY the category name.

User Query:
"{user_input}"
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0,  # Deterministic routing
        messages=[
            {"role": "system", "content": "You are an intent classification system."},
            {"role": "user", "content": routing_prompt}
        ]
    )

    category = response.choices[0].message.content.strip().lower()

    # Safety fallback
    if category not in MODEL_CONFIG:
        category = "general"

    return category

In [6]:
def fetch_bitcoin_price():
    # Mock function (replace with real API if needed)
    return "Current Bitcoin price is approximately $52,000 (mock data)."

In [7]:
def process_request(user_input):

    # Step 1: Route to expert
    category = route_prompt(user_input)
    print(f"[Router Selected]: {category}")

    # Step 2: Tool routing check (Bonus)
    if "bitcoin" in user_input.lower() and "price" in user_input.lower():
        return fetch_bitcoin_price()

    # Step 3: Load expert system prompt
    system_prompt = MODEL_CONFIG[category]["system_prompt"]

    # Step 4: Call expert model
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.7,  # More flexible/creative response
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]
    )

    return response.choices[0].message.content

In [8]:
print(process_request("My python script is throwing an IndexError on line 5."))

[Router Selected]: technical
To assist with debugging the IndexError on line 5 of your Python script, I would need to see the code. However, I can guide you through a general approach to identify and resolve the issue.

1. **Review the line causing the error**: Look at the code on line 5. Identify what operation is being performed. Typically, an IndexError occurs when trying to access an element in a list or other sequence that does not exist (e.g., accessing an index that is out of range).

2. **Check indexing**: Ensure that the index you are trying to access is within the bounds of the list. Remember that Python uses zero-based indexing, meaning the first element is at index 0, and the last element is at index `length - 1`.

3. **Verify data structure**: Make sure the variable you are trying to index is indeed a list or another type of sequence that supports indexing.

Here is a basic example of how an IndexError might occur and how to fix it:

```python
# Example of an IndexError
my

In [9]:
print(process_request("I was charged twice for my subscription this month."))

[Router Selected]: billing
I'm so sorry to hear that you were charged twice for your subscription this month. I can imagine how frustrating that must be for you. Please know that I'm here to help resolve this issue as quickly as possible.

To get started, could you please provide me with some more information about the duplicate charge? This includes the date of the charges, the amount, and the method of payment used. Additionally, do you have your subscription account information or order number handy? This will help me to locate your account and investigate the issue further.

Once I have this information, I'll be happy to look into this matter and work with you to process a refund for the duplicate charge. You can expect a full refund for the incorrect charge, and I'll also review your account to ensure that this doesn't happen again in the future.

If you have any questions or concerns, please don't hesitate to ask. I'm here to support you and appreciate your patience as we work to

In [10]:
print(process_request("Tell me a fun fact about space."))

[Router Selected]: general
Here's one that's out of this world: Did you know that there's a giant storm on Jupiter that's been raging for at least 150 years? It's called the Great Red Spot, and it's so big that three Earths could fit inside it. Can you imagine a storm that's been going on for centuries? Mind blown, right?


In [11]:
print(process_request("What is the current price of Bitcoin?"))

[Router Selected]: general
Current Bitcoin price is approximately $52,000 (mock data).
